# Project objective
This notebook studies how agriculture, health, and education indicators relate to Pakistan's economic performance over time.  
We will focus on transparent cleaning, clear visual analysis, and correlation checks (including lag analysis) 

In [ ]:
pip install pandas numpy matplotlib seaborn scikit-learn scipy 

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
from scipy import stats
warnings.filterwarnings('ignore')

# Set style for prettier charts
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

---
## 1. Data Cleaning
This section will focus on filtering and cleaning the raw csv file so that it becomes usable for further analysis. The raw dataset contains over 98,000 rows and 4,300 unique indicators. Our analysis narrows this down to a targeted set of 15 features spanning from 1990 to 2018. This is done via:

* removing unecessary metadata
* handling duplicates
* feature filtering
* reshaping for easier readbility and operations
* handling missing values 





### 1) Load raw data and run sanity checks
We will load the CSV, remove the metadata row, convert numeric fields, and confirm basic shape/range.  
This ensures the dataset is in a valid format before any analysis.

In [ ]:

raw = pd.read_csv("indicators_pak.csv", low_memory=False)

# Remove metadata row that starts with #country+name
raw = raw[raw["Country Name"] != "#country+name"].copy()

# Convert numeric columns
raw["Year"] = pd.to_numeric(raw["Year"], errors="coerce")
raw["Value"] = pd.to_numeric(raw["Value"], errors="coerce")

# Keep valid numeric rows only
raw = raw.dropna(subset=["Year", "Value"]).copy()
raw["Year"] = raw["Year"].astype(int)

# Keep Pakistan only (safety check)
raw = raw[raw["Country Name"] == "Pakistan"].copy()

print("Rows:", len(raw))
print("Year range:", raw["Year"].min(), "to", raw["Year"].max())
print("Unique indicators:", raw["Indicator Code"].nunique())
print(raw.head())


### 2) Handle duplicates correctly
Before pivoting, we will check duplicate quality.  
If duplicates have same value, I remove them safely. If conflicting duplicates exist, that would need special handling.


In [ ]:

exact_dups = raw.duplicated().sum()

conflicting_groups = (
    raw.groupby(["Year", "Indicator Code"])["Value"]
    .nunique()
    .gt(1)
    .sum()
)

print("Exact duplicate rows:", int(exact_dups))
print("Conflicting duplicate groups:", int(conflicting_groups))

# Safe dedup (conflicting groups are expected to be 0 in your file)
df = raw.drop_duplicates().copy()

print("Rows after dedup:", len(df))


### 3) Select indicators by code (stable and reproducible)
We use World Bank indicator codes instead of long names to avoid matching errors.  
We also include food imports and constant GDP per capita to match project questions better.


In [ ]:

indicator_map = {
    "Agri_GDP_Share": ("NV.AGR.TOTL.ZS", "Agriculture, forestry, and fishing, value added (% of GDP)"),
    "Agri_Employment": ("SL.AGR.EMPL.ZS", "Employment in agriculture (% of total employment) (modeled ILO estimate)"),
    "Cereal_Yield": ("AG.YLD.CREL.KG", "Cereal yield (kg per hectare)"),
    "Agri_Land_Pct": ("AG.LND.AGRI.ZS", "Agricultural land (% of land area)"),
    "Food_Imports": ("TM.VAL.FOOD.ZS.UN", "Food imports (% of merchandise imports)"),
    "Life_Expectancy": ("SP.DYN.LE00.IN", "Life expectancy at birth, total (years)"),
    "Child_Mortality": ("SH.DYN.MORT", "Mortality rate, under-5 (per 1,000 live births)"),
    "DPT_Immunization": ("SH.IMM.IDPT", "Immunization, DPT (% of children ages 12-23 months)"),
    "Literacy_Rate": ("SE.ADT.LITR.ZS", "Literacy rate, adult total (% of people ages 15 and above)"),
    "Primary_Enrollment": ("SE.PRM.ENRR", "School enrollment, primary (% gross)"),
    "Secondary_Enrollment": ("SE.SEC.ENRR", "School enrollment, secondary (% gross)"),
    "Edu_Expenditure": ("SE.XPD.TOTL.GD.ZS", "Government expenditure on education, total (% of GDP)"),
    "GDP_Per_Capita_Const": ("NY.GDP.PCAP.KD", "GDP per capita (constant 2010 US$)"),
    "GDP_Per_Capita_Current": ("NY.GDP.PCAP.CD", "GDP per capita (current US$)"),
    "GDP_Growth": ("NY.GDP.MKTP.KD.ZG", "GDP growth (annual %)")
}

selected_codes = [v[0] for v in indicator_map.values()]
df_sel = df[df["Indicator Code"].isin(selected_codes)].copy()


print("Selected rows:", len(df_sel))
print("Selected indicators found:", df_sel["Indicator Code"].nunique())




### 4) Reshape to year-wise analysis table
We pivot to wide format (rows = year, columns = indicators), then keep 1990-2018 for fair overlap.  
This period has much better indicator coverage than very early and very recent years.


In [ ]:

# Keep one row per year-indicator after sorting
df_sel = (
    df_sel.sort_values(["Year", "Indicator Code"])
    .drop_duplicates(["Year", "Indicator Code"], keep="last")
)

wide = df_sel.pivot(index="Year", columns="Indicator Code", values="Value")

code_to_short = {v[0]: k for k, v in indicator_map.items()}
wide = wide.rename(columns=code_to_short).sort_index()

# Reindex ensures all years appear, even if some indicators are missing
wide = wide.reindex(range(1990, 2019))

coverage = wide.notna().sum().to_frame("non_null")
coverage["missing"] = len(wide) - coverage["non_null"]
coverage["coverage_pct"] = (coverage["non_null"] / len(wide) * 100).round(1)
wide.columns.name = None #indicator code tab was still appearinh

print ("\nModified Dataset: ")
print(wide.head())
print("Shape:", wide.shape)
print("\nCoverage table:")
print(coverage.sort_values("coverage_pct"))
wide.to_csv("cleaned_pak_indicators_dataset.csv", index=True) #for local copy

In [ ]:
import os
os.listdir()
#checking if excel file saved

### 5) Missing-value strategy
We then fill only small gaps for relatively smooth indicators and keep sparse education fields mostly as observed.  
This avoids creating artificial trends in weakly observed series.


In [ ]:

wide_clean = wide.copy()

# Only small-gap interpolation for smoother indicators
small_gap_cols = [
    "Agri_GDP_Share", "Agri_Employment", "Cereal_Yield", "Agri_Land_Pct",
    "Life_Expectancy", "Child_Mortality", "DPT_Immunization",
    "GDP_Per_Capita_Const", "GDP_Per_Capita_Current", "GDP_Growth", "Food_Imports"
]

for col in small_gap_cols:
    if col in wide_clean.columns:
        wide_clean[col] = wide_clean[col].interpolate(method="linear", limit=2)

print("Remaining missing values:")
print(wide_clean.isna().sum().sort_values(ascending=False))

plt.figure(figsize=(12, 4))
sns.heatmap(wide_clean.isna(), cbar=False)
plt.title("Missingness Map (1990-2018)")
plt.xlabel("Indicators")
plt.ylabel("Year")
plt.show()



---
## 2. Exploratory Data Analysis
In this section we will now utlize our cleaned dataset in order to draw conclusions and search for patterns 

### 1) Trend analysis (time-series storytelling)
These plots show sector evolution over time and how each sector tracks economic indicators.  
This is the main narrative view before statistical comparison.


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
fig.suptitle("Pakistan Development Trends (1990-2018)", fontsize=16, fontweight="bold")

# Agriculture
ax = axes[0, 0]
ax.plot(wide_clean.index, wide_clean["Agri_GDP_Share"], marker="o", label="Agri % of GDP")
ax.plot(wide_clean.index, wide_clean["Agri_Employment"], marker="s", label="Agri Employment %")
ax.set_title("Agriculture")
ax.set_ylabel("Percent")
ax.legend()

# Health
ax = axes[0, 1]
ax.plot(wide_clean.index, wide_clean["Life_Expectancy"], marker="o", color="tab:red", label="Life Expectancy")
ax2 = ax.twinx()
ax2.plot(wide_clean.index, wide_clean["Child_Mortality"], marker="s", color="tab:brown", linestyle="--", label="Child Mortality")
ax.set_title("Health")
ax.set_ylabel("Years", color="tab:red")
ax2.set_ylabel("Deaths per 1,000", color="tab:brown")
ax.legend(loc="upper left")
ax2.legend(loc="upper right")

# Education
ax = axes[1, 0]
ax.plot(wide_clean.index, wide_clean["Primary_Enrollment"], marker="o", label="Primary Enrollment")
ax.plot(wide_clean.index, wide_clean["Secondary_Enrollment"], marker="s", label="Secondary Enrollment")
ax.scatter(wide_clean.index, wide_clean["Literacy_Rate"], s=70, label="Literacy Rate")
ax.set_title("Education")
ax.set_ylabel("Percent")
ax.legend()

# Economy
ax = axes[1, 1]
ax.plot(wide_clean.index, wide_clean["GDP_Per_Capita_Const"] / 1000, marker="o", color="tab:purple", label="GDP per capita (constant, $1000s)")
ax2 = ax.twinx()
ax2.plot(wide_clean.index, wide_clean["GDP_Growth"], marker="s", color="tab:orange", label="GDP Growth %")
ax.set_title("Economy")
ax.set_ylabel("GDP per capita ($1000s)", color="tab:purple")
ax2.set_ylabel("GDP Growth %", color="tab:orange")
ax.legend(loc="upper left")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()




### 2) Correlation analysis (levels vs year-to-year changes)
We compute correlations in two ways:  
1) Levels (overall long-term alignment)  
2) First differences (year-to-year co-movement, less trend bias)


In [ ]:
corr_cols = [
    "Agri_GDP_Share", "Agri_Employment", "Cereal_Yield", "Agri_Land_Pct", "Food_Imports",
    "Life_Expectancy", "Child_Mortality", "DPT_Immunization",
    "Literacy_Rate", "Primary_Enrollment", "Secondary_Enrollment", "Edu_Expenditure",
    "GDP_Per_Capita_Const", "GDP_Growth"
]
corr_cols = [c for c in corr_cols if c in wide_clean.columns]

level_df = wide_clean[corr_cols]
diff_df = level_df.diff()

corr_levels = level_df.corr(min_periods=12)
corr_diff = diff_df.corr(min_periods=10)

# Pairwise sample-size matrix for masking weak overlaps
valid = level_df.notna().astype(int)
n_matrix = valid.T.dot(valid)

mask = np.triu(np.ones_like(corr_levels, dtype=bool)) | (n_matrix.values < 12)

plt.figure(figsize=(10, 7))
sns.heatmap(
    corr_levels, mask=mask, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    annot=True, fmt=".2f", linewidths=0.5
)
plt.title("Correlation Matrix (Levels, min 12 overlapping years)")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

for target in ["GDP_Per_Capita_Const", "GDP_Growth"]:
    if target in corr_levels.columns:
        print(f"\nLevel correlations with {target}:")
        print(corr_levels[target].drop(target).sort_values(ascending=False).round(3))
    if target in corr_diff.columns:
        print(f"\nFirst-difference correlations with {target}:")
        print(corr_diff[target].drop(target).sort_values(ascending=False).round(3))


### 3) Sector vs economy scatter plots
These plots show relationship shape, outliers, and time progression.  
We use one consistent colormap so the shared year colorbar is correct.


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Sector Performance vs Economy (GDP per capita, constant)", fontsize=15, fontweight="bold")

norm = plt.Normalize(vmin=wide_clean.index.min(), vmax=wide_clean.index.max())
scatter_handle = None

def scatter_with_trend(ax, x_col, x_label, title):
    global scatter_handle
    temp = wide_clean[[x_col, "GDP_Per_Capita_Const"]].dropna()
    if len(temp) == 0:
        ax.set_title(title + " (no data)")
        return

    scatter_handle = ax.scatter(
        temp[x_col], temp["GDP_Per_Capita_Const"] / 1000,
        c=temp.index, cmap="viridis", norm=norm,
        s=90, alpha=0.8, edgecolor="black"
    )

    if len(temp) >= 3:
        z = np.polyfit(temp[x_col], temp["GDP_Per_Capita_Const"] / 1000, 1)
        p = np.poly1d(z)
        x_sorted = np.sort(temp[x_col].values)
        ax.plot(x_sorted, p(x_sorted), "r--", linewidth=2)
        r = temp[x_col].corr(temp["GDP_Per_Capita_Const"])
        ax.text(0.05, 0.93, f"r = {r:.3f}", transform=ax.transAxes,
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

    ax.set_xlabel(x_label)
    ax.set_ylabel("GDP per capita ($1000s, constant)")
    ax.set_title(title)

scatter_with_trend(axes[0], "Agri_GDP_Share", "Agriculture % of GDP", "Agriculture vs Economy")
scatter_with_trend(axes[1], "Life_Expectancy", "Life Expectancy (years)", "Health vs Economy")
scatter_with_trend(axes[2], "Primary_Enrollment", "Primary Enrollment (%)", "Education vs Economy")

if scatter_handle is not None:
    cbar = fig.colorbar(scatter_handle, ax=axes, fraction=0.025, pad=0.04)
    cbar.set_label("Year")

plt.tight_layout()
plt.show()


### 4) Lag analysis for supporting questions
Correlation at lag helps test whether indicator changes happen before, after, or together with GDP growth.  
Positive lag means the indicator leads GDP growth by that many years.


In [ ]:
def lag_correlation_table(df_in, x_col, y_col, max_lag=5):
    rows = []
    for lag in range(-max_lag, max_lag + 1):
        x_shift = df_in[x_col].shift(lag)
        temp = pd.concat([x_shift, df_in[y_col]], axis=1).dropna()
        r = temp.iloc[:, 0].corr(temp.iloc[:, 1]) if len(temp) >= 8 else np.nan
        rows.append({"lag_years": lag, "correlation": r, "n_pairs": len(temp)})
    return pd.DataFrame(rows)

lag_indicators = ["Agri_GDP_Share", "Food_Imports", "Life_Expectancy", "Primary_Enrollment"]
lag_indicators = [c for c in lag_indicators if c in wide_clean.columns]

plt.figure(figsize=(10, 5))

for col in lag_indicators:
    tbl = lag_correlation_table(wide_clean, col, "GDP_Growth", max_lag=5)
    plt.plot(tbl["lag_years"], tbl["correlation"], marker="o", label=col)

    best = tbl.loc[tbl["correlation"].abs().idxmax()]
    print(f"{col}: best lag = {int(best['lag_years'])}, r = {best['correlation']:.3f}, pairs = {int(best['n_pairs'])}")

plt.axhline(0, color="black", linewidth=1)
plt.axvline(0, color="gray", linestyle="--")
plt.title("Lag Correlation with GDP Growth")
plt.xlabel("Lag in years (positive = indicator leads GDP)")
plt.ylabel("Correlation")
plt.legend()
plt.tight_layout()
plt.show()


### 5) Volatility and Shock Analysis
In this section, we analyze the "stability" of different sectors. Economic growth in Pakistan is often affected by shocks (floods, political changes, global oil prices). We use **Violin Plots** to visualize the distribution of Year-over-Year (YoY) percentage changes. 
* A **wide/fat** violin indicates high volatility (unpredictable growth).
* A **narrow** violin indicates steady, incremental progress.

In [ ]:
# Calculate percentage changes to measure volatility
volatility_df = wide_clean.pct_change().dropna() * 100

# Select representative columns for each sector
cols_to_plot = {
    'GDP_Growth': 'Economy',
    'Agri_GDP_Share': 'Agriculture',
    'Life_Expectancy': 'Health',
    'Primary_Enrollment': 'Education'
}
plot_data = volatility_df[list(cols_to_plot.keys())].rename(columns=cols_to_plot)

plt.figure(figsize=(12, 6))
sns.violinplot(data=plot_data, palette="Set2", inner="quartile")
plt.title("Growth Volatility by Sector (Annual % Change Distribution)", fontsize=14)
plt.ylabel("Annual % Change")
plt.grid(axis='y', alpha=0.3)
plt.show()

### 6) Feature Engineering: Human Capital vs. Economy
To simplify our analysis, we combine **Health** (Life Expectancy) and **Education** (Primary Enrollment) into a single **Human Capital Index**. We normalize these values between 0 and 1 so they can be averaged. We then use a **Linear Regression** line to see how strongly this combined "Human Capital" tracks with GDP per Capita.

In [ ]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

wide_clean['Human_Capital_Index'] = (normalize(wide_clean['Life_Expectancy']) + 
                               normalize(wide_clean['Primary_Enrollment'])) / 2

# Plotting Regression - Using corrected name 'GDP_Per_Capita_Current'
plt.figure(figsize=(10, 6))
sns.regplot(x='Human_Capital_Index', y='GDP_Per_Capita_Current', data=wide_clean, 
            scatter_kws={'s':50, 'alpha':0.6}, line_kws={'color':'red'})

plt.title("Correlation: Human Capital Index vs. GDP Per Capita", fontsize=14)
plt.xlabel("Human Capital Index (Combined Health & Education)")
plt.ylabel("GDP Per Capita (Current US$)")
plt.grid(True, alpha=0.3)
plt.show()


### 7) Final Comparison: Indexed Growth (Base Year = 1990)
To put everything in perspective, we look at how much each sector has grown relative to where it started in 1990. By setting the 1990 value of every indicator to **100**, we can see which sector has seen the most "explosive" growth over the nearly 30-year period.

In [ ]:

growth_cols = ['GDP_Per_Capita_Current', 'Life_Expectancy', 'Agri_GDP_Share', 'Primary_Enrollment']
valid_growth_cols = [c for c in growth_cols if c in wide_clean.columns]

indexed_df = wide_clean[valid_growth_cols].copy()
indexed_df = (indexed_df / indexed_df.iloc[0]) * 100

plt.figure(figsize=(14, 7))
for col in valid_growth_cols:
    plt.plot(indexed_df.index, indexed_df[col], label=col, linewidth=2.5)

plt.axhline(100, color='black', linestyle='--', alpha=0.5) # The 1990 Base Line
plt.title("Pakistan: Comparative Sector Growth Since 1990 (Indexed to 100)", fontsize=15)
plt.ylabel("Growth Index (1990 = 100)")
plt.xlabel("Year")
plt.legend(loc='upper left')
plt.grid(True, alpha=0.2)
plt.show()

**Conclusion:**
In this phase of the project, we transformed a large and noisy dataset of over 98,000 rows into a focused analysis of Pakistan’s key development indicators from 1990–2018. Several clear patterns emerged from the analysis.

Human capital shows the most stability.
Our Human Capital Index — combining health and education indicators — shows a strong and consistent relationship with GDP per capita. While economic growth fluctuates year to year, improvements in life expectancy and school enrollment follow a much steadier upward trend.

Agriculture remains important but volatile.
Although agriculture is a foundational sector of Pakistan’s economy, our analysis shows that its indicators display greater variability over time. This suggests that agricultural performance is more sensitive to external factors such as climate conditions, water availability, and global market changes.

Economic growth is erratic, but development continues.
The indexed growth comparison highlights a key distinction: GDP growth varies significantly across years, while indicators like life expectancy and primary enrollment improve more consistently. This suggests that long-term social progress can continue even during periods of economic instability.

Final Verdict:
While agriculture remains vital for employment and food security, health and education indicators provide a more stable reflection of Pakistan’s long-term economic development and structural progress.

---
## 3. Statistical Inference: Testing Our Observations
In the EDA section, we mostly looked for patterns with charts and descriptive summaries. Now we want to test a few of those patterns more formally
The goal of this is to conclude whether **these differences and relationships strong enough that they probably are not just random variation?**

We will use tools like **p-values** (a rough check of whether a result could happen by chance), **confidence intervals** (a range of plausible averages), and **R-squared** (how much variation a regression explains), but we will explain them in plain language as we go.


### 1) Getting a small side-table ready
Before we run inference and machine learning we will need a few extra indicators that were not kept in the first cleaned table. We are not changing the earlier workflow at all. Just building a small side-table and joining only the extra columns we need for the new sections.


In [ ]:
from IPython.display import display
from scipy import stats
from scipy.stats import pearsonr, spearmanr, ttest_ind, mannwhitneyu
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.preprocessing import StandardScaler

# Group note: we only pull in the extra indicators that were not already part of wide_clean.
supplement_indicator_map = {
    'Gross_Savings': ('NY.GNS.ICTR.GN.ZS', 'Gross savings (% of GNI)'),
    'Adj_Net_Savings': ('NY.ADJ.SVNG.GN.ZS', 'Adjusted net savings (% of GNI)'),
    'Inflation': ('FP.CPI.TOTL.ZG', 'Inflation, consumer prices (annual %)'),
    'Exports_GDP': ('NE.EXP.GNFS.ZS', 'Exports of goods and services (% of GDP)'),
    'Imports_GDP': ('NE.IMP.GNFS.ZS', 'Imports of goods and services (% of GDP)')
}

pretty_labels = {
    'Agri_GDP_Share': 'Agriculture share of GDP (%)',
    'Agri_Employment': 'Agricultural employment (%)',
    'Cereal_Yield': 'Cereal yield (kg per hectare)',
    'Agri_Land_Pct': 'Agricultural land (% of land area)',
    'Food_Imports': 'Food imports (% of merchandise imports)',
    'Life_Expectancy': 'Life expectancy (years)',
    'Child_Mortality': 'Child mortality (per 1,000)',
    'DPT_Immunization': 'DPT immunization (%)',
    'Literacy_Rate': 'Literacy rate (%)',
    'Primary_Enrollment': 'Primary enrollment (%)',
    'Secondary_Enrollment': 'Secondary enrollment (%)',
    'Edu_Expenditure': 'Education expenditure (% of GDP)',
    'GDP_Per_Capita_Const': 'GDP per capita (constant 2010 US$)',
    'GDP_Per_Capita_Current': 'GDP per capita (current US$)',
    'GDP_Growth': 'GDP growth (%)',
    'Gross_Savings': 'Gross savings (% of GNI)',
    'Adj_Net_Savings': 'Adjusted net savings (% of GNI)',
    'Inflation': 'Inflation (%)',
    'Exports_GDP': 'Exports (% of GDP)',
    'Imports_GDP': 'Imports (% of GDP)'
}

short_labels = {
    'Agri_GDP_Share': 'Agri GDP share',
    'Cereal_Yield': 'Cereal yield',
    'Life_Expectancy': 'Life expectancy',
    'DPT_Immunization': 'DPT immunization',
    'Primary_Enrollment': 'Primary enrollment',
    'Literacy_Rate': 'Literacy rate',
    'GDP_Per_Capita_Const': 'GDP per capita',
    'GDP_Growth': 'GDP growth',
    'Gross_Savings': 'Gross savings',
    'Adj_Net_Savings': 'Adjusted net savings',
    'Inflation': 'Inflation',
    'Exports_GDP': 'Exports/GDP',
    'Imports_GDP': 'Imports/GDP'
}

supplement_codes = [value[0] for value in supplement_indicator_map.values()]
supplement_df = (
    df[df['Indicator Code'].isin(supplement_codes)]
    .sort_values(['Year', 'Indicator Code'])
    .drop_duplicates(['Year', 'Indicator Code'], keep='last')
    .copy()
)

supplement_wide = supplement_df.pivot(index='Year', columns='Indicator Code', values='Value')
supplement_wide = supplement_wide.rename(columns={value[0]: key for key, value in supplement_indicator_map.items()})
supplement_wide = supplement_wide.sort_index().reindex(range(1990, 2019))
supplement_wide.columns.name = None

analysis_wide = wide_clean.copy().join(supplement_wide, how='left')
analysis_wide = analysis_wide.sort_index().reindex(range(1990, 2019))
analysis_wide.index.name = 'Year'

# Group note: we only interpolate very short gaps so that we do not create an artificial trend.
for col in supplement_indicator_map:
    if col in analysis_wide.columns:
        analysis_wide[col] = analysis_wide[col].interpolate(method='linear', limit=2, limit_direction='both')

extra_coverage = analysis_wide[list(supplement_indicator_map.keys())].notna().sum().to_frame('non_null_years')
extra_coverage['coverage_pct'] = (extra_coverage['non_null_years'] / len(analysis_wide) * 100).round(1)
extra_coverage.index = [pretty_labels[idx] for idx in extra_coverage.index]

print('Extra indicator coverage in the 1990-2018 study window:')
display(extra_coverage)


### 2) Correlation Check: Do these indicators move together?
We will check both **Pearson correlation** (best for straight-line relationships) and **Spearman correlation** (best for rank-based or monotonic relationships, meaning the variables still move in the same general direction even if the line is not perfectly straight).

We picked four pairs that match the themes of our project: 
* wealth and health 
* savings behavior
* the education pipeline
* the idea of structural transformation away from agriculture


In [ ]:
correlation_pairs = [
    ('GDP_Per_Capita_Const', 'Life_Expectancy', 'Wealth and health'),
    ('Gross_Savings', 'Adj_Net_Savings', 'Savings behavior'),
    ('Primary_Enrollment', 'Literacy_Rate', 'Education pipeline'),
    ('Agri_GDP_Share', 'GDP_Growth', 'Structural transformation signal')
]

def correlation_takeaway(pearson_r, pearson_p, spearman_r):
    if np.isnan(pearson_r):
        return 'Not enough overlapping years to judge this pair well.'
    if pearson_p < 0.05 and abs(pearson_r) >= 0.7:
        return 'A strong relationship shows up clearly in the data.'
    if pearson_p < 0.05 and abs(pearson_r) >= 0.4:
        return 'There is a moderate signal, but it is not perfect.'
    if abs(spearman_r) >= 0.5:
        return 'The ranking still moves together even if the line is not very tight.'
    return 'This pair looks fairly weak, so we should stay cautious.'

correlation_rows = []
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
axes = axes.flatten()

for ax, (x_col, y_col, story_name) in zip(axes, correlation_pairs):
    pair_df = analysis_wide[[x_col, y_col]].dropna().copy()

    pearson_r, pearson_p = pearsonr(pair_df[x_col], pair_df[y_col])
    spearman_r, spearman_p = spearmanr(pair_df[x_col], pair_df[y_col])

    correlation_rows.append({
        'Pair': f'{short_labels[x_col]} vs {short_labels[y_col]}',
        'n': len(pair_df),
        'Pearson r': pearson_r,
        'Pearson p-value': pearson_p,
        'Spearman rho': spearman_r,
        'Spearman p-value': spearman_p,
        'Interpretation': correlation_takeaway(pearson_r, pearson_p, spearman_r)
    })

    sns.regplot(
        data=pair_df,
        x=x_col,
        y=y_col,
        ax=ax,
        scatter_kws={'s': 70, 'alpha': 0.8, 'edgecolor': 'black'},
        line_kws={'color': '#d1495b', 'linewidth': 2}
    )

    ax.set_title(story_name, fontsize=12, fontweight='bold')
    ax.set_xlabel(pretty_labels[x_col])
    ax.set_ylabel(pretty_labels[y_col])
    ax.text(
        0.03,
        0.97,
        f'Pearson r = {pearson_r:.2f}\nSpearman rho = {spearman_r:.2f}\nn = {len(pair_df)}',
        transform=ax.transAxes,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9)
    )

plt.suptitle('Do these move together? Linear and monotonic correlation checks', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

correlation_summary_df = pd.DataFrame(correlation_rows)
numeric_cols = ['Pearson r', 'Pearson p-value', 'Spearman rho', 'Spearman p-value']
correlation_summary_df[numeric_cols] = correlation_summary_df[numeric_cols].round(3)
display(correlation_summary_df)
print('A high correlation does not prove causation, but it does tell us which pairs are worth exploring further.')


### 3) Did things actually change after 2010, or could this be random variation?
A lot of our trend plots suggested that the later years look different from the 1990s. To test that more carefully, we compare **Pre-2000** (1990-1999) with **Post-2010** (2010-2018).

We chose **Welch's t-test** because our two era samples are small and their spreads might not be identical. We also use the **Mann-Whitney U test** as a non-parametric backup, which just means it does not depend as heavily on the data being normally shaped.


In [ ]:
comparison_cols = ['GDP_Per_Capita_Const', 'Life_Expectancy', 'DPT_Immunization', 'Gross_Savings', 'Adj_Net_Savings']
era_summary_rows = []

def era_takeaway(mean_diff, t_p, u_p):
    if np.isnan(t_p):
        return 'There is not enough data for a reliable era test.'
    if t_p < 0.05 and u_p < 0.05 and mean_diff > 0:
        return 'The later era looks clearly higher in both tests.'
    if t_p < 0.05 and mean_diff < 0:
        return 'The later era looks clearly lower in the t-test.'
    if t_p < 0.05:
        return 'There is evidence of a difference, but we should still be careful.'
    return 'This difference could easily be ordinary noise.'

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, col in zip(axes, comparison_cols):
    pre_values = analysis_wide.loc[1990:1999, col].dropna()
    post_values = analysis_wide.loc[2010:2018, col].dropna()

    pre_mean = pre_values.mean()
    post_mean = post_values.mean()
    mean_diff = post_mean - pre_mean

    pre_sem = pre_values.std(ddof=1) / np.sqrt(len(pre_values)) if len(pre_values) > 1 else np.nan
    post_sem = post_values.std(ddof=1) / np.sqrt(len(post_values)) if len(post_values) > 1 else np.nan

    t_stat, t_p = ttest_ind(pre_values, post_values, equal_var=False, nan_policy='omit')
    u_stat, u_p = mannwhitneyu(pre_values, post_values, alternative='two-sided')

    era_summary_rows.append({
        'Indicator': short_labels[col],
        'Pre-2000 mean': pre_mean,
        'Post-2010 mean': post_mean,
        'Mean difference': mean_diff,
        'Welch p-value': t_p,
        'Mann-Whitney p-value': u_p,
        'Pre n': len(pre_values),
        'Post n': len(post_values),
        'Interpretation': era_takeaway(mean_diff, t_p, u_p)
    })

    ax.bar(
        ['Pre-2000', 'Post-2010'],
        [pre_mean, post_mean],
        yerr=[pre_sem, post_sem],
        capsize=6,
        color=['#8da0cb', '#66c2a5'],
        edgecolor='black'
    )
    ax.set_title(short_labels[col], fontsize=11, fontweight='bold')
    ax.set_ylabel(pretty_labels[col])

for ax in axes[len(comparison_cols):]:
    ax.axis('off')

plt.suptitle('Pre-2000 vs Post-2010 means with standard-error bars', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

era_summary_df = pd.DataFrame(era_summary_rows)
era_numeric_cols = ['Pre-2000 mean', 'Post-2010 mean', 'Mean difference', 'Welch p-value', 'Mann-Whitney p-value']
era_summary_df[era_numeric_cols] = era_summary_df[era_numeric_cols].round(3)
display(era_summary_df)


### P-Value Interpretation: 

If the p val is small then it provides some evidence that the era difference is real. If it is not small thenthat does not mean "nothing happened". It just means the difference is not strong enough for us to sound overly confident with such a small yearly sample.


### 4) Confidence Intervals: How certain are we about recent values?
We now examine the most recent 5-year window, **2014-2018**. A **95% confidence interval** gives a plausible range for the true recent average. We use the **t-distribution** instead of the z-distribution because the sample is small so we want the more conservative small-sample version.


In [ ]:
recent_start_year = 2014
recent_end_year = 2018
ci_indicators = ['GDP_Per_Capita_Const', 'Life_Expectancy', 'DPT_Immunization']
ci_rows = []

for col in ci_indicators:
    recent_values = analysis_wide.loc[recent_start_year:recent_end_year, col].dropna()
    n = len(recent_values)
    mean_value = recent_values.mean()
    std_value = recent_values.std(ddof=1)
    std_error = std_value / np.sqrt(n) if n > 1 else np.nan
    critical_value = stats.t.ppf(0.975, df=n - 1) if n > 1 else np.nan
    margin_of_error = critical_value * std_error if n > 1 else np.nan

    ci_rows.append({
        'Indicator': short_labels[col],
        'Years used': n,
        'Recent mean': mean_value,
        'CI lower': mean_value - margin_of_error,
        'CI upper': mean_value + margin_of_error,
        'Margin of error': margin_of_error
    })

ci_summary_df = pd.DataFrame(ci_rows)
ci_numeric_cols = ['Recent mean', 'CI lower', 'CI upper', 'Margin of error']
ci_summary_df[ci_numeric_cols] = ci_summary_df[ci_numeric_cols].round(3)
display(ci_summary_df)

plt.figure(figsize=(10, 5))
plt.errorbar(
    x=ci_summary_df['Indicator'],
    y=ci_summary_df['Recent mean'],
    yerr=ci_summary_df['Margin of error'],
    fmt='o',
    capsize=7,
    linewidth=2,
    color='#1f78b4'
)
plt.title(f'95% confidence intervals for recent averages ({recent_start_year}-{recent_end_year})')
plt.xlabel('Indicator')
plt.ylabel('Estimated recent average')
plt.tight_layout()
plt.show()


### 4.1) What does this mean?

We are 95% confident that the true recent average lies inside each interval, but that comes with the assumption that the recent period has to be reasonably representative. Wider intervals indicate that yearly development data can be **noisy**, so we should not over-interpret tiny differences.


---
## 4. Machine Learning: Modeling the Sector–Economy Link
Now that we have explored patterns and tested some of them statistically, we want to use machine learning to directly answer our **primary research question**: 
- which of **agriculture**, **health**, or **education** shows the strongest relationship with Pakistan's economy over time?

We will do this in 4 steps: 
* Build a multidimensional linear regression on sector indicators
* Apply **regularization** (Ridge and Lasso) to handle the fact that development indicators tend to move together
* Evaluate using **k-fold cross-validation**
* Try a **logistic regression** to see whether the same indicators can also classify a year as a **high-growth** or **low-growth year**


### 1) Problem framing and feature setup
Our regression target is **GDP per capita (constant 2010 US$)**. We pick **two indicators per sector** so that no sector dominates the regression by sheer feature count:

- **Agriculture:** Cereal yield, Agricultural employment
- **Health:** Life expectancy, DPT immunization
- **Education:** Primary enrollment, Education expenditure

We chose two indicators per sector so each sector enters the model with **equal voice** otherwise the model would have been biased toward "finding" a specific sector as the strongest simply by feature count.

We **standardize** every predictor using a standard scalar method so that a coefficient of 1 means the same thing in every column. Without this step, Cereal yield (measured in thousands of kg/hectare) would dwarf Education expenditure (measured as a single-digit percentage) just because of unit scale.

We also fill the few remaining missing years using linear interpolation. We acknowledge this introduces a small modeling assumption, but the alternative is dropping rows in an already small (n=29) sample.


In [ ]:
from sklearn.linear_model import Ridge, Lasso, LogisticRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

ml_features = {
    'Cereal_Yield': 'Agriculture',
    'Agri_Employment': 'Agriculture',
    'Life_Expectancy': 'Health',
    'DPT_Immunization': 'Health',
    'Primary_Enrollment': 'Education',
    'Edu_Expenditure': 'Education',
}
ml_target = 'GDP_Per_Capita_Const'

# Aggressive interpolation (used only for the ML section, with small N)
ml_data = wide_clean[list(ml_features.keys()) + [ml_target, 'GDP_Growth']].copy()
for col in ml_data.columns:
    ml_data[col] = ml_data[col].interpolate(method='linear', limit_direction='both')
ml_data = ml_data.dropna()

X_ml = ml_data[list(ml_features.keys())]
y_ml = ml_data[ml_target]

feature_table = pd.DataFrame({
    'Feature': X_ml.columns,
    'Sector': [ml_features[c] for c in X_ml.columns]
})

print(f"ML dataset shape: {ml_data.shape}")
print(f"Years covered: {ml_data.index.min()} to {ml_data.index.max()}")
print('\nFeature plan:')
display(feature_table)


### 2) Multidimensional Linear Regression
A **multidimensional linear regression** fits a single model where GDP per capita is approximated as a weighted sum of all six sector indicators plus an intercept (the bias). 

After standardizing the predictors, each **coefficient** tells us how many dollars of GDP per capita are associated with a one-standard-deviation increase in that indicator while the other indicators fixed.

The bigger the coefficient (in absolute value), the stronger that indicator's link with GDP per capita relative to the other indicators in the model. 
The sign tells us whether the relationship is positive or negative.


In [ ]:
ols_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
ols_pipeline.fit(X_ml, y_ml)

ols_coefficients = pd.DataFrame({
    'Feature': X_ml.columns,
    'Sector': [ml_features[c] for c in X_ml.columns],
    'Coefficient': ols_pipeline.named_steps['model'].coef_
}).sort_values('Coefficient', key=abs, ascending=False).reset_index(drop=True)

ols_train_r2 = ols_pipeline.score(X_ml, y_ml)
print(f"Training R-squared: {ols_train_r2:.3f}")
print(f"Intercept (mean GDP per capita on standardized features): {ols_pipeline.named_steps['model'].intercept_:.1f}")
print('\nStandardized coefficients (ranked by absolute size):')
display(ols_coefficients.round(2))

sector_palette = {'Agriculture': '#66c2a5', 'Health': '#fc8d62', 'Education': '#8da0cb'}

plt.figure(figsize=(10, 5))
sns.barplot(
    data=ols_coefficients,
    x='Coefficient',
    y='Feature',
    hue='Sector',
    palette=sector_palette,
    dodge=False
)
plt.axvline(0, color='black', linewidth=1)
plt.title('Standardized OLS coefficients by sector')
plt.xlabel('Coefficient (effect of 1-std change on GDP per capita)')
plt.tight_layout()
plt.show()


### 2.1) Interpretation:
The training R-squared looks high but a model can fit the training data well and still fail on new data and our 29 yearly observations make overfitting very plausible. 

On top of that, reality dicatates that sector indicators tend to move together: countries that improve their cereal yields also tend to improve life expectancy and enrollment in the same years for instance. 

When predictors move in lockstep, ordinary regression can generate **inflated and unstable coefficients**. 

The next step is to address both problems with **regularization**.


### 3) Bias-Variance Tradeoff and Regularization
A model can go wrong in two opposite ways:

- **High bias:** means the model is too simple to capture the real pattern. Leads to **underfitting**
- **High variance:** means the model is so flexible that it fits the noise in the training data. AKA **overfitting**

The cure for high variance is **regularization**: we add a penalty to the model that shrinks coefficients toward zero. We try two types:

- **Ridge regression** adds an L2 penalty (sum of squared coefficients). This shrinks every coefficient smoothly toward zero, but rarely sets any to exactly zero.
- **Lasso regression** adds an L1 penalty (sum of absolute coefficients). It also shrinks coefficients a lot more drastically so many coefficients become **exactly zero**, which means Lasso doubles as automatic **feature selection**.

We evaluate across several values of the penalty strength `alpha` to see how each model behaves. The features that Lasso keeps last (as `alpha` grows) are the strongest predictors since they **carry information that no other feature can replace**.


In [ ]:
ridge_alphas = [0.1, 1.0, 10.0, 50.0]
lasso_alphas = [1.0, 10.0, 50.0, 100.0]

regularization_rows = []

# Baseline OLS for reference
regularization_rows.append({
    'Model': 'OLS',
    'alpha': 0,
    'Train R^2': ols_train_r2,
    **dict(zip(X_ml.columns, ols_pipeline.named_steps['model'].coef_))
})

for alpha in ridge_alphas:
    pipe = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=alpha))])
    pipe.fit(X_ml, y_ml)
    regularization_rows.append({
        'Model': 'Ridge',
        'alpha': alpha,
        'Train R^2': pipe.score(X_ml, y_ml),
        **dict(zip(X_ml.columns, pipe.named_steps['model'].coef_))
    })

for alpha in lasso_alphas:
    pipe = Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=alpha, max_iter=20000))])
    pipe.fit(X_ml, y_ml)
    regularization_rows.append({
        'Model': 'Lasso',
        'alpha': alpha,
        'Train R^2': pipe.score(X_ml, y_ml),
        **dict(zip(X_ml.columns, pipe.named_steps['model'].coef_))
    })

regularization_df = pd.DataFrame(regularization_rows).round(2)
print('Coefficient comparison across regularization strengths:')
display(regularization_df)

# Lasso path: how each coefficient evolves as alpha increases
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

lasso_path_alphas = np.logspace(-1, 2.2, 40)
lasso_path_coefs = []
for a in lasso_path_alphas:
    pipe = Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=a, max_iter=20000))])
    pipe.fit(X_ml, y_ml)
    lasso_path_coefs.append(pipe.named_steps['model'].coef_)
lasso_path_coefs = np.array(lasso_path_coefs)

for j, feat in enumerate(X_ml.columns):
    axes[0].plot(lasso_path_alphas, lasso_path_coefs[:, j], label=feat,
                 color=sector_palette[ml_features[feat]],
                 linestyle='-' if j % 2 == 0 else '--', linewidth=2)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_xscale('log')
axes[0].set_xlabel('alpha (penalty strength)')
axes[0].set_ylabel('Lasso coefficient')
axes[0].set_title('Lasso coefficient path')
axes[0].legend(loc='best', fontsize=8)

# True resilience = the largest alpha at which this feature is still non-zero
resilience_rows = []
for j, feat in enumerate(X_ml.columns):
    nonzero_mask = np.abs(lasso_path_coefs[:, j]) > 1e-6
    if nonzero_mask.any():
        max_alpha_alive = lasso_path_alphas[nonzero_mask].max()
    else:
        max_alpha_alive = 0
    resilience_rows.append({
        'Feature': feat,
        'Sector': ml_features[feat],
        'Survives until alpha =': max_alpha_alive
    })
features_by_resilience = (pd.DataFrame(resilience_rows)
    .sort_values('Survives until alpha =', ascending=False)
    .reset_index(drop=True))

sns.barplot(
    data=features_by_resilience,
    x='Survives until alpha =',
    y='Feature',
    hue='Sector',
    palette=sector_palette,
    dodge=False,
    ax=axes[1]
)
axes[1].set_xscale('log')
axes[1].set_title('Feature resilience: largest alpha\nat which the coefficient stays non-zero')
axes[1].set_xlabel('Highest alpha where feature survives (log scale)')

plt.tight_layout()
plt.show()

print('\nFeatures ranked by Lasso resilience (last to be zeroed out as penalty grows):')
display(features_by_resilience.round(2))


### 3.2) Which regularization is the most informative?

The Lasso path is the most informative chart. As we increase the penalty `alpha`, more and more coefficients are pulled to exactly zero. As mentioned before, the features that **survive longest** are the ones carrying information no other feature can replace and that is what we treat as the answer to our primary research question.

### 3.3) Reading the surviving-feature ranking ###

**Agriculture (Cereal yield) and Education (Primary enrollment)** are the most resilient. Health indicators get shrunk away earlier, which does *not* mean health is unimportant just it means health's information overlaps heavily with the other features (so dropping it doesn't hurt much). This is a more honest answer than just looking at raw correlations where every upward-trending series correlates strongly with every other upward-trending series. 


### 4) Cross-Validation: how well does the model generalize?
The training R-squared we just saw measures fit on the data the model already saw. To estimate how the model would behave on **unseen** years, we use **k-fold cross-validation** as following:

* Split the dataset into 5 folds
* Hold one fold out for testing 
* Train on the other 4, score, and rotate until all the fold permutations have been evaluated. 
* Report the mean and the spread for each fold. 

A model with high mean R-squared and low spread is a model we trust. We compare three setups: plain OLS, Ridge with a moderate penalty, and Lasso with a moderate penalty.


In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'OLS': Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'Ridge (alpha=1)': Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'Ridge (alpha=10)': Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=10.0))]),
    'Lasso (alpha=10)': Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=10.0, max_iter=20000))]),
    'Lasso (alpha=50)': Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=50.0, max_iter=20000))]),
}

cv_rows = []
for name, pipe in cv_models.items():
    fold_scores = cross_val_score(pipe, X_ml, y_ml, cv=kfold, scoring='r2')
    cv_rows.append({
        'Model': name,
        'Mean CV R^2': fold_scores.mean(),
        'Std across folds': fold_scores.std(),
        'Min fold R^2': fold_scores.min(),
        'Max fold R^2': fold_scores.max()
    })

cv_results_df = pd.DataFrame(cv_rows).round(3)
display(cv_results_df)

plt.figure(figsize=(10, 5))
plt.bar(
    cv_results_df['Model'],
    cv_results_df['Mean CV R^2'],
    yerr=cv_results_df['Std across folds'],
    capsize=6,
    color='#1f78b4',
    edgecolor='black'
)
plt.axhline(0, color='black', linewidth=1)
plt.ylabel('Cross-validated R-squared')
plt.title('5-fold cross-validation comparison')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()


### 4.1) Interpretation
Plain OLS still does well, but the regularized models match or slightly beat it with **smaller, more interpretable coefficients**. This is the **bias-variance tradeoff** at work: we accepted a tiny extra bit of bias (regularization makes coefficients smaller than they should be) and got back lower variance (the model is steadier across folds).

### 4.2) The key takeaway 
Long-run sector indicators predict GDP per capita level wuite well (cross-validated R-squared in the 0.9 range). That is a strong signal that the relationship is structural as opposed to being noise


### 5) Logistic Regression: classifying high-growth years
So far we have predicted a continuous quantity (GDP per capita). Now we ask whether the same sector indicators tell us whether a given year was a **high-growth year** (GDP growth above the median) or a **low-growth year** (below the median)?

### 5.1) This is a classification problem now
The natural tool is **logistic regression**. Instead of predicting a number directly, logistic regression outputs a probability between 0 and 1 — the probability that the year falls into the "high-growth" class. We then classify the year as high-growth when that probability exceeds 0.5.

We use the **median** as our cutoff so that both classes have roughly the same number of years. This makes the baseline accuracy 50%, which is the bar a useful classifier has to clear.


In [ ]:
median_growth = ml_data['GDP_Growth'].median()
print(f"Median GDP growth (cutoff): {median_growth:.2f}%")

clf_X = ml_data[list(ml_features.keys())]
clf_y = (ml_data['GDP_Growth'] > median_growth).astype(int)

print(f"Class balance: {clf_y.mean():.2f} high-growth, {1 - clf_y.mean():.2f} low-growth")

logistic_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000))
])
logistic_pipeline.fit(clf_X, clf_y)

train_predictions = logistic_pipeline.predict(clf_X)
train_accuracy = accuracy_score(clf_y, train_predictions)
cv_accuracy = cross_val_score(logistic_pipeline, clf_X, clf_y, cv=kfold, scoring='accuracy')

print(f"\nTraining accuracy: {train_accuracy:.3f}")
print(f"5-fold CV accuracy: {cv_accuracy.mean():.3f} (std {cv_accuracy.std():.3f})")
print(f"Baseline (always predict majority class): {max(clf_y.mean(), 1 - clf_y.mean()):.3f}")

logistic_coefs_df = pd.DataFrame({
    'Feature': clf_X.columns,
    'Sector': [ml_features[c] for c in clf_X.columns],
    'Coefficient': logistic_pipeline.named_steps['model'].coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False).reset_index(drop=True)

cm = confusion_matrix(clf_y, train_predictions)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted Low', 'Predicted High'],
    yticklabels=['Actual Low', 'Actual High'],
    ax=axes[0], cbar=False
)
axes[0].set_title('Confusion matrix (training data)')

sns.barplot(
    data=logistic_coefs_df,
    x='Coefficient',
    y='Feature',
    hue='Sector',
    palette=sector_palette,
    dodge=False,
    ax=axes[1]
)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('Logistic regression coefficients\n(positive = pushes toward high-growth class)')

plt.tight_layout()
plt.show()

display(logistic_coefs_df.round(2))


### 5.2) Why is our result seemingly random? A finding on Pakistan's Data:
The cross-validated accuracy hovers near the 50% baseline, which means the classifier is barely better than guessing. A model that can predict GDP per capita *level* with R-squared above 0.9 cannot reliably say whether a given year had above-median growth.

This result is indicative of the data when **contextualized specifically to Pakistan**. Yearly **growth rates** are noisy and driven by short-run shocks (floods, oil price moves, political events) that our slow-moving sector indicators cannot see. The link between development sectors and the economy lives mostly in the **long run** and not in year-to-year swings.

This connects all the way back to the **volatility plots in the EDA section**: GDP growth had a much wider violin than life expectancy or enrollment. Now we have made that observation precise, short-run growth is essentially unpredictable from our indicator set while long-run levels are still highly predictable.


### 6) One Insightful Visualization
For the final takeaway, we combine the regularization story with the time-series story:
- We pick the two features that survived Lasso longest
- Plot them alongside GDP per capita over time 
- Show the same relationship as a scatter with a regression line.

If the same two indicators (one agricultural, one educational) really do drive the long-run alignment, we should see them tracking GDP per capita closely over the entire 1990–2018 window.


In [ ]:
# Top 2 surviving features by absolute Lasso coefficient at the highest alpha
top_two_features = features_by_resilience.head(2)['Feature'].tolist()

# Indexed plot (1990 = 100) so the three curves are comparable
indexed_top = ml_data[top_two_features + [ml_target]].copy()
indexed_top = (indexed_top / indexed_top.iloc[0]) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: indexed time series
for col in top_two_features:
    axes[0].plot(indexed_top.index, indexed_top[col], marker='o',
                 label=f"{col} ({ml_features[col]})",
                 color=sector_palette[ml_features[col]], linewidth=2.5)
axes[0].plot(indexed_top.index, indexed_top[ml_target],
             marker='s', label=ml_target, color='black', linewidth=2.5)
axes[0].axhline(100, color='gray', linestyle='--', alpha=0.6)
axes[0].set_title('Top Lasso-surviving sector indicators vs GDP per capita\n(indexed, 1990 = 100)')
axes[0].set_ylabel('Growth index (1990 = 100)')
axes[0].set_xlabel('Year')
axes[0].legend()

# Right: scatter of the single top feature vs GDP, regression line, colored by year
top_feature = top_two_features[0]
scatter_df = ml_data[[top_feature, ml_target]].copy()
scatter_df['Year'] = scatter_df.index

scat = axes[1].scatter(
    scatter_df[top_feature], scatter_df[ml_target],
    c=scatter_df['Year'], cmap='viridis', s=110, edgecolor='black'
)
z = np.polyfit(scatter_df[top_feature], scatter_df[ml_target], 1)
xs = np.linspace(scatter_df[top_feature].min(), scatter_df[top_feature].max(), 100)
axes[1].plot(xs, np.poly1d(z)(xs), color='red', linestyle='--', linewidth=2, label='OLS fit')
axes[1].set_xlabel(top_feature)
axes[1].set_ylabel(ml_target)
axes[1].set_title(f'{top_feature} vs GDP per capita (1990–2018)')
axes[1].legend()
plt.colorbar(scat, ax=axes[1], label='Year')

plt.tight_layout()
plt.show()


### 6.1) Final Interpretation
The two indicators that survived Lasso the longest move alongside GDP per capita with remarkable consistency. The scatter on the right makes the same point in a different way: every year sits close to the fitted line, and the color gradient (year) climbs neatly from the bottom-left to the top-right corner. The relationship is structural and persistent over the entire timeframe

---
## 5. Conclusion: What We Learned (and What We'd Do Next)
Putting EDA, statistical inference, and machine learning together, three pictures lined up.

**On the primary research question**, our regularized regression gave a clearer answer than raw correlations could. After Lasso shrunk away the redundant features, **agriculture (Cereal yield) and education (Primary enrollment)** were the two indicators that carried unique, non-replaceable information about Pakistan's GDP per capita. Health indicators had high raw correlations with GDP, but most of that information was already captured by the agricultural and educational signals. So while all three sectors track the economy, agriculture and education appear to track it most independently.

**On long-run vs short-run dynamics**, the contrast between the regression and the classification is the most striking thing we found. The regression on levels achieved cross-validated R-squared above 0.9, while logistic classification of high-growth versus low-growth years barely beat random guessing. Long-run alignment is real and strong; short-run yearly swings are mostly driven by shocks our indicator set cannot see. This matches what the volatility plots already hinted at in the EDA.

**On methodology**, working through bias-variance tradeoff and regularization made one thing concrete for us: with only 29 yearly observations and six predictors that move together, plain OLS gives reassuringly high R-squared but unstable coefficients. Cross-validation and Lasso were the right tools for this dataset size, and they gave us a model whose conclusions we can defend.

We also want to stay honest about the limitations. Our sample is small, education indicators required interpolation, and yearly economic data is autocorrelated in ways that make any t-test slightly optimistic. Correlation never proves causation, even with regularized regression. If we had more time, we would test lagged predictors more carefully, bring in provincial data to expand the sample, and try time-series models that respect temporal structure (ARIMA, dynamic regression) instead of treating years as independent observations.

The biggest lesson for us as a group was methodological rather than topical. Picking the right model is not the hard part since almost every model gave us a coefficient on every predictor. The hard part was choosing methods that fit the size and structure of our data, and being honest when a model failed to explain something. The classification result felt like a setback at first but ended up being the cleanest finding of the project: **structural and yearly dynamics of an economy are different things**, and not every method will see both.
